# Architectural Control Experiment -- Small-Scale BERT vs. Mamba

**Purpose**: Prove that the geometric stability gap between Transformers and SSMs
is **architectural**, not an artifact of genomic/proteomic training data or any
specific foundation model's training recipe.

## Design

Train a **small BERT** (~2M params) and a **small Mamba** (~2M params) from scratch
on three synthetic continuous datasets that have nothing to do with biology:

| Dataset | Dynamics | Analogy |
|---------|----------|---------|
| Superposed sine waves | Smooth, periodic | Baseline -- easiest |
| Damped harmonic oscillator | Decaying, directional | Intermediate -- time asymmetric |
| Lorenz attractor | Chaotic, sensitive | Hardest -- SDIC stress test |

All sequences are discretized into **256 bins** (token vocabulary), trained with
**Causal Language Modeling** (next-token prediction, identical to GPT), and evaluated with the same
`StabilityHarness` (shesha-geometry) used in every other experiment.

## Perturbations (analogous to DNA experiments)

- **Value noise** at 1%, 2%, 5%, 10% of positions (analogous to SNP mutation rates)
- **Time reversal** (analogous to reverse complement)

## Setup Instructions

1. Upload `evaluation_harness.py` to `/content/`
2. Change Runtime to **GPU** (Runtime > Change runtime type > GPU)
3. Run cells in order (~60 min total on A100)

---

In [ ]:
# Install Dependencies

print('Installing core dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

# Build mamba-ssm from source (one-time ~10 min compile)
print('\nBuilding mamba-ssm from source (this takes ~10 min, go get coffee)...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

import os
import sys
sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/architectural_control_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Dataset parameters ---
SEQ_LEN    = 512       # sequence length (time steps)
N_BINS     = 256       # discretization bins (vocabulary size)
N_TRAIN    = 50_000    # training sequences per dataset
N_EVAL     = 2_000     # evaluation sequences per dataset
SEED       = 320
SEEDS      = [320, 1991, 9, 7258, 7]  # 5 seeds suffice for architectural ablation

# --- Special tokens ---
PAD_TOKEN  = N_BINS     # index 256
MASK_TOKEN = N_BINS + 1 # index 257
VOCAB_SIZE = N_BINS + 2 # 258 total (256 bins + PAD + MASK)

# --- Architecture parameters (matched at ~2M params) ---
D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 4   # BERT only
FFN_DIM    = 1024  # BERT only
D_STATE    = 16  # Mamba only
D_CONV     = 4   # Mamba only
EXPAND     = 2   # Mamba only

# --- Training parameters ---
LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64

# --- Perturbation rates (analogous to SNP rates in DNA experiments) ---
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]

# --- Dataset names ---
DATASETS = ['waveform', 'oscillator', 'lorenz']
ARCHITECTURES = ['SmallBERT', 'SmallMamba', 'SmallStripedHyena']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Vocab size: {VOCAB_SIZE} (256 bins + MASK + PAD)')
print(f'Sequence length: {SEQ_LEN}')
print(f'Training: {N_TRAIN} sequences x {EPOCHS} epochs, batch={BATCH_SIZE}')
print(f'Datasets: {DATASETS}')
print(f'Architectures: {ARCHITECTURES}')
print(f'\nOutput: {OUTPUT_BASE}')

In [ ]:
# Synthetic Dataset Generators
#
# Three continuous-signal datasets, each producing 1D sequences of length
# SEQ_LEN, discretized into N_BINS integer tokens in [0, 255].
#
# discretize() uses dataset-global min/max so the same physical
# state maps to the same bin across all sequences. Two-pass generation:
#   Pass 1: generate raw continuous trajectories, compute global range
#   Pass 2: discretize all trajectories using the global range

import numpy as np
from scipy.integrate import solve_ivp


def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    '''Map continuous values to integer bins in [0, n_bins-1].

    Uses dataset-global min/max when provided so the same physical
    state maps to the same bin across all sequences. Per-sequence
    normalization destroys cross-sequence comparability.
    '''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    bins = np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)
    return bins


# =====================================================================
# Dataset 1: Superposed Sine Waves
# =====================================================================
def generate_waveforms(n_sequences, seq_len=SEQ_LEN, seed=SEED,
                       global_range=None):
    '''Superposition of 3-5 sine waves with random freq/amp/phase.

    Returns (sequences, global_range) where global_range = (min, max).
    '''
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, seq_len, endpoint=False)

    # Pass 1: generate raw signals
    raw_signals = []
    for _ in range(n_sequences):
        n_sines = rng.integers(3, 6)
        signal = np.zeros(seq_len)
        for _ in range(n_sines):
            freq  = rng.uniform(0.5, 50.0)
            amp   = rng.uniform(0.1, 1.0)
            phase = rng.uniform(0, 2 * np.pi)
            signal += amp * np.sin(2 * np.pi * freq * t + phase)
        raw_signals.append(signal)

    # Compute global range
    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    # Pass 2: discretize with global range
    sequences = [discretize(s, global_min=global_range[0], global_max=global_range[1])
                 for s in raw_signals]
    return np.array(sequences, dtype=np.int64), global_range


# =====================================================================
# Dataset 2: Damped Harmonic Oscillator
# =====================================================================
def generate_oscillators(n_sequences, seq_len=SEQ_LEN, seed=SEED,
                         global_range=None):
    '''x(t) = A * exp(-gamma*t) * cos(omega*t + phi)

    Returns (sequences, global_range).
    '''
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)

    raw_signals = []
    for _ in range(n_sequences):
        A     = rng.uniform(0.5, 2.0)
        gamma = rng.uniform(0.2, 2.0)
        omega = rng.uniform(2.0, 20.0)
        phi   = rng.uniform(0, 2 * np.pi)
        signal = A * np.exp(-gamma * t) * np.cos(omega * t + phi)
        raw_signals.append(signal)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    sequences = [discretize(s, global_min=global_range[0], global_max=global_range[1])
                 for s in raw_signals]
    return np.array(sequences, dtype=np.int64), global_range


# =====================================================================
# Dataset 3: Lorenz Attractor (x-coordinate)
# =====================================================================
def _lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]


def generate_lorenz(n_sequences, seq_len=SEQ_LEN, seed=SEED,
                    global_range=None, return_continuous=False):
    '''Integrate Lorenz system from random ICs, extract x-coordinate.

    Returns (sequences, global_range) or (sequences, global_range, raw_trajectories)
    if return_continuous=True.
    '''
    rng = np.random.default_rng(seed)
    t_span = (0, 25)
    t_eval = np.linspace(*t_span, seq_len)

    raw_signals = []
    for _ in range(n_sequences):
        x0 = rng.uniform(-15, 15)
        y0 = rng.uniform(-15, 15)
        z0 = rng.uniform(10, 40)
        sol = solve_ivp(
            _lorenz_rhs, t_span, [x0, y0, z0],
            t_eval=t_eval, method='RK45', max_step=0.05,
        )
        if sol.success and len(sol.y[0]) == seq_len:
            raw_signals.append(sol.y[0])
        else:
            x0 += rng.uniform(-1, 1)
            sol2 = solve_ivp(
                _lorenz_rhs, t_span, [x0, y0, z0],
                t_eval=t_eval, method='RK45', max_step=0.01,
            )
            sig = sol2.y[0][:seq_len]
            if len(sig) < seq_len:
                sig = np.pad(sig, (0, seq_len - len(sig)), mode='edge')
            raw_signals.append(sig)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    sequences = [discretize(s, global_min=global_range[0], global_max=global_range[1])
                 for s in raw_signals]

    result = (np.array(sequences, dtype=np.int64), global_range)
    if return_continuous:
        result = result + (raw_signals,)
    return result


# =====================================================================
# Unified generator
# =====================================================================
GENERATORS = {
    'waveform':   generate_waveforms,
    'oscillator': generate_oscillators,
    'lorenz':     generate_lorenz,
}

# Storage for global ranges (train ranges reused for eval)
GLOBAL_RANGES = {}

# Quick sanity check
for name, gen_fn in GENERATORS.items():
    sample, grange = gen_fn(5, seed=1991)
    assert sample.shape == (5, SEQ_LEN), f'{name}: bad shape {sample.shape}'
    assert sample.min() >= 0 and sample.max() < N_BINS, f'{name}: out of range'
    print(f'{name:12s}: shape={sample.shape}, range=[{sample.min()}, {sample.max()}], '
          f'global_range=({grange[0]:.2f}, {grange[1]:.2f})')

print('\nAll 3 dataset generators ready (using dataset-global discretization)')


In [ ]:
# Continuous Signal Perturbation Suite
#
# Analogous to DNAPerturbationSuite from the genomic experiments:
#   - Value noise at 1/2/5/10%  =  analogous to SNP mutations
#   - Time reversal             =  analogous to reverse complement

from dataclasses import dataclass, field


@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: np.ndarray   # (n_seq, seq_len) int64
    params: dict = field(default_factory=dict)
    description: str = ''


def noise_perturb(sequences, rate, rng, n_bins=N_BINS):
    """Replace `rate` fraction of positions with noisy neighbouring bins.

    For each position selected (with probability `rate`), add Gaussian
    noise to the bin index (std = 10% of bin range = 25.6 bins) and
    clip to [0, n_bins-1].  This is the continuous-signal analogue of
    a single-nucleotide polymorphism.
    """
    out = sequences.copy()
    noise_std = n_bins * 0.10   # 25.6 bins
    mask = rng.random(out.shape) < rate
    noise = rng.normal(0, noise_std, size=out.shape).astype(np.int64)
    out[mask] = np.clip(out[mask] + noise[mask], 0, n_bins - 1)
    return out


def time_reverse(sequences):
    """Reverse every sequence along the time axis."""
    return sequences[:, ::-1].copy()


class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES

    def run_all(self, sequences):
        """Generate all perturbations.  Returns dict[str, PerturbedSet]."""
        results = {}

        for rate in self.noise_rates:
            name = f'noise_{int(rate * 100)}pct'
            results[name] = PerturbedSet(
                name=name,
                category='noise',
                sequences=noise_perturb(sequences, rate, self.rng),
                params={'noise_rate': rate},
                description=f'Value noise at {rate*100:.0f}% of positions',
            )

        results['time_reversal'] = PerturbedSet(
            name='time_reversal',
            category='reversal',
            sequences=time_reverse(sequences),
            params={},
            description='Time reversal (analogue of reverse complement)',
        )

        return results


# Quick check
_suite = ContinuousPerturbationSuite(seed=1991)
_sample, _ = generate_waveforms(10, seed=1991)
_perts = _suite.run_all(_sample)
print(f'Perturbation types: {list(_perts.keys())}')
for k, v in _perts.items():
    assert v.sequences.shape == _sample.shape
    print(f'  {k:16s}: range=[{v.sequences.min()}, {v.sequences.max()}]')
del _suite, _sample, _perts
print('\nContinuousPerturbationSuite ready')

In [ ]:
# Model Definitions -- SmallBERT (Causal), SmallMamba, SmallStripedHyena
#
# SmallBERT uses a causal attention mask (like GPT), NOT bidirectional.
# This ensures the autoregressive evaluation is valid.
#
# Both architectures matched at ~2M parameters:
#   - Same token embedding (257 vocab -> 256 dim)
#   - Same positional embedding (512 positions -> 256 dim)
#   - Same prediction head (256 dim -> 257 logits)
#   - 4 layers each, d_model = 256

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# =====================================================================
# SmallBERT -- Causal Transformer Decoder (GPT-style)
# =====================================================================
class SmallBERT(nn.Module):
    """4-layer Transformer with CAUSAL attention mask (GPT-style decoder).
    
    NOTE: This is NOT a bidirectional encoder. We apply a causal mask so
    each position can only attend to itself and previous positions.
    This makes autoregressive generation valid."""

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _causal_mask(self, seq_len, device):
        """Generate causal (upper-triangular) attention mask."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

    def forward(self, x, return_hidden=False):
        """x: (batch, seq_len) int64 token indices.
        Returns logits (batch, seq_len, vocab) and optionally hidden (batch, seq_len, d_model)."""
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        
        # Apply causal mask so position i can only attend to positions <= i
        causal_mask = self._causal_mask(L, x.device)
        h = self.encoder(h, mask=causal_mask)
        h = self.norm(h)
        logits = self.head(h)
        if return_hidden:
            return logits, h
        return logits


# =====================================================================
# SmallMamba -- SSM (using mamba-ssm or pure-PyTorch fallback)
# =====================================================================
if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock

    class SmallMamba(nn.Module):
        """4-layer Mamba for Causal Language Modeling (native CUDA backend)."""

        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)

            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(
                    MambaBlock(d_model=d_model, d_state=d_state,
                               d_conv=d_conv, expand=expand)
                )
                self.norms.append(nn.LayerNorm(d_model))

            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)

            self._init_weights()

        def _init_weights(self):
            for name, p in self.named_parameters():
                if 'tok_emb' in name or 'pos_emb' in name:
                    nn.init.normal_(p, std=0.02)
                elif p.dim() > 1 and 'head' in name:
                    nn.init.xavier_uniform_(p)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))   # pre-norm residual
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits

else:
    # ---- Pure-PyTorch SSM Fallback (simplified S4-style) ----
    class SimpleSSMLayer(nn.Module):
        """A simplified selective state-space layer in pure PyTorch."""

        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
            self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                      padding=d_conv - 1, groups=d_inner)
            self.x_proj   = nn.Linear(d_inner, d_state * 2, bias=False)
            self.dt_proj  = nn.Linear(d_state, d_inner, bias=True)
            self.A_log    = nn.Parameter(torch.log(torch.randn(d_inner, d_state).abs() + 1e-4))
            self.D        = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)

        def forward(self, x):
            B, L, D = x.shape
            xz = self.in_proj(x)
            x_inner, z = xz.chunk(2, dim=-1)
            x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :L].transpose(1, 2)
            x_conv = torch.silu(x_conv)
            y = x_conv * torch.silu(z)
            y = y * self.D.unsqueeze(0).unsqueeze(0)
            return self.out_proj(y)

    class SmallMamba(nn.Module):
        """4-layer SSM for Causal Language Modeling (pure PyTorch fallback)."""

        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)

            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(
                    SimpleSSMLayer(d_model, d_state, d_conv, expand)
                )
                self.norms.append(nn.LayerNorm(d_model))

            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits

    print('NOTE: Using pure-PyTorch SSM fallback (no mamba-ssm)')


# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        # No weight tying. vocab_size (258) != d_model (256),
        # so tying head.weight = tok_emb.weight causes shape mismatch.
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


# =====================================================================
# Print parameter counts
# =====================================================================
MODEL_CLASSES = {
    'SmallBERT': SmallBERT,
    'SmallMamba': SmallMamba,
    'SmallStripedHyena': SmallStripedHyena,
}

for name, cls in MODEL_CLASSES.items():
    m = cls()
    n_params = sum(p.numel() for p in m.parameters())
    print(f'{name:12s}: {n_params/1e6:.2f}M params')
    del m

print('\nModel definitions ready (both use CAUSAL language modeling)')
if not MAMBA_AVAILABLE:
    print('  (SmallMamba uses pure-PyTorch SSM fallback)')





In [ ]:
# Training Loop -- Causal Language Modeling (CLM)
#
# Both SmallBERT and SmallMamba now predict x_{t+1} given x_0...x_t.
# This aligns training with the autoregressive evaluation.

import time
from torch.utils.data import DataLoader, TensorDataset


def create_causal_batch(x):
    """Create CLM-style shifted input/target pairs.
    
    Input:  x_0, x_1, ..., x_{L-2}  (all tokens except the last)
    Target: x_1, x_2, ..., x_{L-1}  (all tokens except the first)
    
    Args:
        x: (batch, seq_len) int64 original token indices
    
    Returns:
        inputs:  (batch, seq_len-1) 
        targets: (batch, seq_len-1)
    """
    inputs  = x[:, :-1]   # everything except the last token
    targets = x[:, 1:]    # everything except the first token
    return inputs, targets


def train_model(model, train_data, val_data, arch_name, dataset_name,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                weight_decay=WEIGHT_DECAY, seed=SEED):
    """Train a model on causal language modeling (next-token prediction).
    
    Args:
        model: nn.Module with forward(x) -> (batch, seq_len, vocab)
        train_data: np.ndarray (n_train, seq_len) int64
        val_data: np.ndarray (n_val, seq_len) int64
        arch_name: str for logging
        dataset_name: str for logging
        seed: random seed for this run
    
    Returns:
        model (best checkpoint), train_losses, val_losses
    """
    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    model = model.to(DEVICE)

    train_ds = TensorDataset(torch.from_numpy(train_data).long())
    val_ds   = TensorDataset(torch.from_numpy(val_data).long())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,
                                  weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader)
    )
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state = None
    train_losses = []
    val_losses = []

    ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_seed{seed}_best.pt'

# Check for existing checkpoint
    if os.path.exists(ckpt_path):
        print(f'  Loading existing checkpoint: {ckpt_path}')
        try:
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE,
                                             weights_only=True))
            return model, [], []
        except RuntimeError as e:
            print(f'  Checkpoint shape mismatch, retraining: {e}')
            os.remove(ckpt_path)
            print(f'  Deleted stale checkpoint: {ckpt_path}')

    print(f'  Training {arch_name} on {dataset_name} (seed={seed}, {epochs} epochs)...')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for (batch_x,) in train_loader:
            batch_x = batch_x.to(DEVICE)
            inputs, targets = create_causal_batch(batch_x)

            logits = model(inputs)  # (B, L-1, V)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_train = epoch_loss / n_batches
        train_losses.append(avg_train)

        # Validation
        model.eval()
        val_loss = 0.0
        val_batches = 0
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(DEVICE)
                inputs, targets = create_causal_batch(batch_x)
                logits = model(inputs)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                val_loss += loss.item()
                val_batches += 1

        avg_val = val_loss / max(val_batches, 1)
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            elapsed = time.time() - start
            print(f'    Epoch {epoch+1:3d}/{epochs}  '
                  f'train={avg_train:.4f}  val={avg_val:.4f}  '
                  f'best_val={best_val_loss:.4f}  [{elapsed:.0f}s]')

    # Restore best and save
    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)

    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min  (best val loss: {best_val_loss:.4f})')
    print(f'  Checkpoint: {ckpt_path}')

    return model, train_losses, val_losses


print('Training loop ready (Causal Language Modeling)')


In [ ]:
# Evaluation Harness (shesha-geometry)

from evaluation_harness import StabilityHarness, ModelReport

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=5,
)

print('Shesha StabilityHarness configured (bootstrap=5)')

In [ ]:
# Run All Conditions (3 datasets x 3 architectures)
#
# Uses dataset-global discretization ranges. Train and eval
# data share the same global range so bins are comparable.

import pandas as pd
import gc


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    '''Extract mean-pooled last-layer hidden states.'''
    model.eval()
    all_embs = []
    n = len(sequences)
    for i in range(0, n, batch_size):
        batch = torch.from_numpy(sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


# =====================================================================
# Main experiment loop
# =====================================================================

print('=' * 70)
print('ARCHITECTURAL CONTROL EXPERIMENT')
print(f'Conditions: {len(DATASETS)} datasets x {len(ARCHITECTURES)} architectures = '
      f'{len(DATASETS) * len(ARCHITECTURES)} conditions')
print('=' * 70)

all_reports = {}
all_detailed_rows = []
training_curves = {}

for ds_name in DATASETS:
    gen_fn = GENERATORS[ds_name]

    print(f"\n{'='*70}")
    print(f'DATASET: {ds_name.upper()}')
    print('=' * 70)

    # Generate data with global discretization
    # train and eval share the same global range
    cache_train = f'{CACHE_DIR}/{ds_name}_train_v2.npy'
    cache_eval  = f'{CACHE_DIR}/{ds_name}_eval_v2.npy'
    cache_range = f'{CACHE_DIR}/{ds_name}_global_range_v2.npy'

    if os.path.exists(cache_train) and os.path.exists(cache_eval):
        train_data = np.load(cache_train)
        eval_data = np.load(cache_eval)
        grange = tuple(np.load(cache_range))
        GLOBAL_RANGES[ds_name] = grange
        print(f'  Loaded cached data: train={train_data.shape}, eval={eval_data.shape}')
        print(f'  Global range: ({grange[0]:.2f}, {grange[1]:.2f})')
    else:
        print(f'  Generating {N_TRAIN} train + {N_EVAL} eval sequences...')
        train_data, grange = gen_fn(N_TRAIN, seed=SEED)
        GLOBAL_RANGES[ds_name] = grange
        # Use SAME global range for eval
        eval_data, _ = gen_fn(N_EVAL, seed=SEED + 1000, global_range=grange)
        np.save(cache_train, train_data)
        np.save(cache_eval, eval_data)
        np.save(cache_range, np.array(grange))
        print(f'  Saved: train={train_data.shape}, eval={eval_data.shape}')
        print(f'  Global range: ({grange[0]:.2f}, {grange[1]:.2f})')

    # Generate perturbations on eval data
    print('  Generating perturbations on eval set...')
    pert_suite = ContinuousPerturbationSuite(seed=SEED)
    perturbed_sets = pert_suite.run_all(eval_data)

    for arch_name in ARCHITECTURES:
        print(f"\n  {'_'*60}")
        print(f'  ARCHITECTURE: {arch_name}')
        print(f'  {"_"*60}')

        condition_key = f'{arch_name}_{ds_name}'

        # 1. Create & train model
        model = MODEL_CLASSES[arch_name]()
        n_params = sum(p.numel() for p in model.parameters()) / 1e6

        model, t_losses, v_losses = train_model(
            model, train_data, eval_data, arch_name, ds_name, seed=SEED
        )

        # Seed ablation: train additional models on Lorenz only
        if ds_name == 'lorenz':
            for ablation_seed in SEEDS:
                if ablation_seed == SEED:
                    continue
                abl_ckpt = f'{CKPT_DIR}/{arch_name}_{ds_name}_seed{ablation_seed}_best.pt'
                if os.path.exists(abl_ckpt):
                    print(f'    Seed {ablation_seed}: checkpoint exists, skipping')
                    continue
                print(f'    Seed ablation: {arch_name} x {ds_name} x seed={ablation_seed}')
                abl_model = MODEL_CLASSES[arch_name]()
                abl_model, _, _ = train_model(
                    abl_model, train_data, eval_data, arch_name, ds_name, seed=ablation_seed
                )
                del abl_model
                cleanup_gpu()
        training_curves[(ds_name, arch_name)] = (t_losses, v_losses)

        # 2. Extract clean embeddings
        cache_clean = f'{CACHE_DIR}/{condition_key}_clean_v2.npy'
        if os.path.exists(cache_clean):
            print(f'  Loading cached clean embeddings')
            embeddings_clean = np.load(cache_clean)
        else:
            print(f'  Extracting clean embeddings ({N_EVAL} sequences)...')
            embeddings_clean = extract_embeddings(model, eval_data)
            np.save(cache_clean, embeddings_clean)
        print(f'    Shape: {embeddings_clean.shape}')

        # 3. Extract perturbed embeddings
        perturbed_embeddings = {}
        for pert_name, pset in perturbed_sets.items():
            cache_pert = f'{CACHE_DIR}/{condition_key}_{pert_name}_v2.npy'
            if os.path.exists(cache_pert):
                perturbed_embeddings[pert_name] = np.load(cache_pert)
            else:
                print(f'    Embedding: {pert_name}...')
                perturbed_embeddings[pert_name] = extract_embeddings(
                    model, pset.sequences
                )
                np.save(cache_pert, perturbed_embeddings[pert_name])

        # 4. Run Shesha evaluation
        print('  Running Shesha evaluation...')
        report = harness.evaluate_all_perturbations(
            model_name=condition_key,
            embeddings_clean=embeddings_clean,
            perturbed_dict=perturbed_embeddings,
        )
        all_reports[(ds_name, arch_name)] = report

        # 5. Collect detailed results
        for pert_name, metrics in report.perturbation_breakdown().items():
            all_detailed_rows.append({
                'dataset': ds_name,
                'architecture': arch_name,
                'n_params_M': round(n_params, 2),
                'perturbation': pert_name,
                'rdm_similarity': metrics.get('rdm_similarity_score', 0),
                'rdm_drift': metrics.get('rdm_drift', 0),
                'pert_stability': metrics.get('perturbation_stability_score', 0),
                'pert_magnitude': metrics.get('perturbation_magnitude', 0),
                'composite': metrics.get('composite_stability', 0),
            })

        summary = report.summary()
        print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
        print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')
        print(f'  Pert. Stability:     {summary["mean_perturbation_stability_score"]:.4f}')

        del model
        cleanup_gpu()

# Save results
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/architectural_control_detailed_v2.csv'
df_detailed.to_csv(detailed_path, index=False)

print(f'\n{"="*70}')
print('ALL CONDITIONS COMPLETE')
print('=' * 70)
print(f'\nDetailed results saved to: {detailed_path}')
print(df_detailed.to_string(index=False))


In [ ]:
# Per-Dataset Comparison -- BERT vs Mamba
#
# One row per dataset (3 rows), three panels each:
#   A. Composite stability per perturbation
#   B. RDM similarity per perturbation
#   C. Perturbation magnitude per perturbation

import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 11})

ARCH_COLORS = {
    'SmallBERT':  '#2563EB',   # blue
    'SmallMamba': '#16A34A',
    'SmallStripedHyena': '#7C3AED',   # green
}
DATASET_LABELS = {
    'waveform':   'Superposed Sine Waves',
    'oscillator': 'Damped Harmonic Oscillator',
    'lorenz':     'Lorenz Attractor',
}

METRIC_COLS = [
    ('composite',      'Composite Stability'),
    ('rdm_similarity', 'RDM Similarity'),
    ('pert_magnitude', 'Perturbation Magnitude'),
]

fig, axes = plt.subplots(len(DATASETS), 3, figsize=(18, 5 * len(DATASETS)))

for row, ds_name in enumerate(DATASETS):
    df_ds = df_detailed[df_detailed['dataset'] == ds_name]
    pert_names = df_ds[df_ds['architecture'] == ARCHITECTURES[0]]['perturbation'].tolist()
    n_perts = len(pert_names)
    x_pos = np.arange(n_perts)
    bar_w = 0.35

    for col, (metric_col, metric_label) in enumerate(METRIC_COLS):
        ax = axes[row, col]

        for i, arch in enumerate(ARCHITECTURES):
            df_a = df_ds[df_ds['architecture'] == arch].sort_values('perturbation')
            vals = df_a[metric_col].values
            offset = (i - 0.5) * bar_w
            ax.bar(x_pos + offset, vals, width=bar_w, alpha=0.8,
                   label=arch, color=ARCH_COLORS[arch],
                   edgecolor='white', linewidth=0.5)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(pert_names, rotation=35, ha='right', fontsize=9)
        ax.set_ylabel(metric_label)
        ax.set_title(f'{DATASET_LABELS[ds_name]} -- {metric_label}', fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.2)

plt.suptitle('Architectural Control: SmallBERT vs SmallMamba\n'
             'Per-Dataset, Per-Perturbation Breakdown',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/per_dataset_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

In [ ]:
# Cross-Dataset Aggregate -- Cross-Dataset Aggregate
#
# Left panel:  Mean composite stability for BERT vs Mamba across all 3 datasets
# Right panel: Overlay with full-scale genomic experiments (if cached results exist)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# =====================================================================
# Panel A: Aggregate across datasets
# =====================================================================
ax = axes[0]

agg = df_detailed.groupby(['dataset', 'architecture']).agg(
    composite_mean=('composite', 'mean'),
    composite_std=('composite', 'std'),
).reset_index()

x_positions = np.arange(len(DATASETS))
n_arch = len(ARCHITECTURES)
bar_width = 0.8 / n_arch

for i, arch in enumerate(ARCHITECTURES):
    df_a = agg[agg['architecture'] == arch].set_index('dataset').loc[DATASETS]
    offset = (i - (n_arch - 1) / 2) * bar_width
    bars = ax.bar(
        x_positions + offset, df_a['composite_mean'],
        yerr=df_a['composite_std'], width=bar_width,
        label=arch, color=ARCH_COLORS[arch], alpha=0.8,
        capsize=5, edgecolor='white', linewidth=0.8,
    )
    # Value labels
    for bar, val in zip(bars, df_a['composite_mean']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x_positions)
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASETS], fontsize=10)
ax.set_ylabel('Mean Composite Stability', fontsize=12)
ax.set_title('A. Architectural Gap Across Datasets', fontweight='bold', fontsize=13)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(0, 1.05)

# Compute and annotate the gap
for ds_idx, ds_name in enumerate(DATASETS):
    bert_val = agg[(agg['dataset'] == ds_name) & (agg['architecture'] == 'SmallBERT')]['composite_mean'].values
    mamba_val = agg[(agg['dataset'] == ds_name) & (agg['architecture'] == 'SmallMamba')]['composite_mean'].values
    if len(bert_val) > 0 and len(mamba_val) > 0:
        gap = mamba_val[0] - bert_val[0]
        mid = (bert_val[0] + mamba_val[0]) / 2
        ax.annotate(
            f'{gap:+.3f}',
            xy=(ds_idx, mid), fontsize=9,
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                      edgecolor='gray', alpha=0.9),
        )

# =====================================================================
# Panel B: Overlay with full-scale results (if available)
# =====================================================================
ax = axes[1]

# Compute control experiment summaries
control_means = {}
for arch in ARCHITECTURES:
    vals = df_detailed[df_detailed['architecture'] == arch]['composite'].values
    control_means[arch] = (vals.mean(), vals.std())

# Try loading full-scale results
full_scale_data = {}
FULL_SCALE_CACHES = {
    'ESM-2 (Transformer)': './results/esm2_scaling_experiment/results/',
    'NT (Transformer)': './results/nt_scaling_experiment/results/',
    'Caduceus (SSM)': './results/caduceus_scaling_experiment/results/',
}

for label, path in FULL_SCALE_CACHES.items():
    try:
        # Try to find the detailed CSV
        for fname in os.listdir(path):
            if fname.endswith('_detailed.csv') and 'full' in fname.lower():
                df_fs = pd.read_csv(os.path.join(path, fname))
                mean_comp = df_fs['composite'].mean()
                std_comp = df_fs['composite'].std()
                full_scale_data[label] = (mean_comp, std_comp)
                break
    except Exception:
        pass

# Build bar chart
all_labels = []
all_means = []
all_stds = []
all_colors = []

# Control results
all_labels.append('SmallBERT\n(Control)')
all_means.append(control_means['SmallBERT'][0])
all_stds.append(control_means['SmallBERT'][1])
all_colors.append(ARCH_COLORS['SmallBERT'])

all_labels.append('SmallMamba\n(Control)')
all_means.append(control_means['SmallMamba'][0])
all_stds.append(control_means['SmallMamba'][1])
all_colors.append(ARCH_COLORS['SmallMamba'])

if 'SmallStripedHyena' in control_means:
    all_labels.append('StripedHyena\n(Control)')
    all_means.append(control_means['SmallStripedHyena'][0])
    all_stds.append(control_means['SmallStripedHyena'][1])
    all_colors.append(ARCH_COLORS.get('SmallStripedHyena', '#7C3AED'))

# Full-scale results
FS_COLORS = {
    'ESM-2 (Transformer)': '#3B82F6',
    'NT (Transformer)':    '#DC2626',
    'Caduceus (SSM)':      '#059669',
}

for label, (m, s) in full_scale_data.items():
    all_labels.append(label)
    all_means.append(m)
    all_stds.append(s)
    all_colors.append(FS_COLORS.get(label, '#6B7280'))

x_pos = np.arange(len(all_labels))
bars = ax.bar(x_pos, all_means, yerr=all_stds, capsize=5,
              color=all_colors, alpha=0.8, edgecolor='white', linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(all_labels, fontsize=9, ha='center')
ax.set_ylabel('Mean Composite Stability', fontsize=12)
ax.set_title('B. Control vs Full-Scale Foundation Models', fontweight='bold', fontsize=13)
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(0, 1.05)

for bar, val in zip(bars, all_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

if not full_scale_data:
    ax.text(0.7, 0.5, '(Full-scale results\nnot yet cached)',
            transform=ax.transAxes, ha='center', va='center',
            fontsize=11, color='gray', style='italic')

plt.suptitle('Architectural Control Experiment\n'
             'The Transformer-SSM Stability Gap is Architectural',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/cross_dataset_aggregate.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()


# =====================================================================
# Statistical test: paired Wilcoxon signed-rank test
# =====================================================================
from scipy.stats import wilcoxon

print('\n' + '=' * 60)
print('STATISTICAL TESTS (paired across perturbation types)')
print('=' * 60)

for arch_a, arch_b in [('SmallBERT', 'SmallMamba'), ('SmallBERT', 'SmallStripedHyena')]:
    # Pair by (dataset, perturbation)
    df_a = df_detailed[df_detailed['architecture'] == arch_a].sort_values(['dataset', 'perturbation'])
    df_b = df_detailed[df_detailed['architecture'] == arch_b].sort_values(['dataset', 'perturbation'])

    if len(df_a) == len(df_b) and len(df_a) > 0:
        vals_a = df_a['composite'].values
        vals_b = df_b['composite'].values
        diff = vals_b - vals_a

        try:
            stat, p_value = wilcoxon(diff, alternative='two-sided')
            mean_diff = np.mean(diff)
            print(f'\n  {arch_b} vs {arch_a}:')
            print(f'    Mean difference: {mean_diff:+.4f} (positive = {arch_b} more stable)')
            print(f'    Wilcoxon signed-rank: W={stat:.1f}, p={p_value:.4f}')
            if p_value < 0.05:
                direction = arch_b if mean_diff > 0 else arch_a
                print(f'    --> Significant at p<0.05: {direction} is more stable')
            else:
                print(f'    --> Not significant at p<0.05')
        except ValueError as e:
            print(f'\n  {arch_b} vs {arch_a}: Could not compute Wilcoxon ({e})')

# =====================================================================
# Summary statistics
# =====================================================================
print('\n' + '=' * 60)
print('AGGREGATE RESULTS')
print('=' * 60)

for arch in ARCHITECTURES:
    m, s = control_means[arch]
    print(f'  {arch:12s}: composite = {m:.4f} +/- {s:.4f}')

bert_m = control_means['SmallBERT'][0]
mamba_m = control_means['SmallMamba'][0]
gap = mamba_m - bert_m
print(f'\n  Gap (Mamba - BERT): {gap:+.4f}')
if gap > 0.01:
    print('  --> SSM (Mamba) is MORE stable than Transformer (BERT)')
elif gap < -0.01:
    print('  --> Transformer (BERT) is MORE stable than SSM (Mamba)')
else:
    print('  --> No significant architectural gap detected')

for label, (m, s) in full_scale_data.items():
    print(f'  {label:25s}: composite = {m:.4f} +/- {s:.4f}')

In [ ]:
# Phase Portrait Reconstruction -- Phase Portrait Reconstruction
#
# 
#   - Ground truth uses actual continued Lorenz integration from the same
#     initial conditions, not a rough approximation from discretized means.
#   - Plotting consolidated into reusable functions (was ~200 lines of duplication).
#   - undiscretize() uses the dataset's global range for proper rescaling.

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def generate_autoregressive(model, seed_seq, steps=1000, temperature=0.8):
    '''Generate autoregressive trajectory from a seed sequence.'''
    model.eval()
    curr_seq = seed_seq.clone()
    generated = []

    with torch.no_grad():
        for _ in range(steps):
            logits = model(curr_seq)
            next_logits = logits[0, -1, :N_BINS] / temperature
            probs = torch.softmax(next_logits, dim=0)
            next_token = torch.multinomial(probs, 1).item()
            generated.append(next_token)
            curr_seq = torch.cat([
                curr_seq[:, 1:],
                torch.tensor([[next_token]], device=curr_seq.device)
            ], dim=1)

    return np.array(generated)


def undiscretize(tokens, global_range=None, n_bins=N_BINS):
    '''Convert bin indices back to continuous values.

    Uses the dataset global range for proper rescaling.
    Without this, the [0,1] range doesn't correspond to physical units.
    '''
    normed = tokens.astype(np.float32) / (n_bins - 1)  # [0, 1]
    if global_range is not None:
        gmin, gmax = global_range
        return normed * (gmax - gmin) + gmin
    return normed


# =====================================================================
# Generate Phase Portraits for Lorenz Dataset
# =====================================================================

dataset_name = 'lorenz'
print('=' * 70)
print(f'PHASE PORTRAIT RECONSTRUCTION: {dataset_name.upper()}')
print('=' * 70)

GENERATION_SEED = 12345
torch.manual_seed(GENERATION_SEED)
np.random.seed(GENERATION_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GENERATION_SEED)
print(f'Generation seed: {GENERATION_SEED}')
print()

# Load checkpoints
models_loaded = {}
for arch_name in ARCHITECTURES:
    ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_seed{SEED}_best.pt'
    if not os.path.exists(ckpt_path):
        print(f'WARNING: Checkpoint not found: {ckpt_path}')
        print('  Skipping phase portrait generation. Train models first.')
        break

    ckpt_state = torch.load(ckpt_path, map_location=DEVICE)
    ckpt_vocab = ckpt_state['tok_emb.weight'].shape[0]
    model = MODEL_CLASSES[arch_name](vocab_size=ckpt_vocab).to(DEVICE)
    model.load_state_dict(ckpt_state)
    models_loaded[arch_name] = model
    print(f'Loaded: {ckpt_path}')

if len(models_loaded) == len(ARCHITECTURES):
    # Generate ground truth with return_continuous=True to get raw trajectories
    lorenz_range = GLOBAL_RANGES.get('lorenz', None)
    print('\nGenerating ground truth Lorenz sequences...')
    gt_sequences, gt_range, gt_raw = generate_lorenz(
        100, seed=SEED + 9999, global_range=lorenz_range, return_continuous=True)

    # Use the ACTUAL raw continuous trajectory as ground truth,
    # not an approximation from discretized bin means
    gt_continuous_raw = gt_raw[0]  # First trajectory, raw continuous values
    # Normalize to [0, 1] using the same global range
    gmin, gmax = gt_range
    gt_trajectory = (gt_continuous_raw - gmin) / (gmax - gmin + 1e-12)

    seed_seq = torch.from_numpy(gt_sequences[0:1]).long().to(DEVICE)
    print(f'  Seed sequence shape: {seed_seq.shape}')
    print(f'  GT trajectory from actual Lorenz integration (not approximated)')

    # Generate long trajectories (autoregressive)
    N_STEPS = 1200
    TEMP = 0.7
    print(f'\nGenerating autoregressive trajectories ({N_STEPS} steps, T={TEMP})...')

    trajectories = {}
    for arch_name, model in models_loaded.items():
        print(f'  {arch_name}...', end=' ')
        traj = generate_autoregressive(model, seed_seq, steps=N_STEPS, temperature=TEMP)
        traj_continuous = undiscretize(traj, global_range=gt_range)
        # Normalize same way as GT
        traj_normed = (traj_continuous - gmin) / (gmax - gmin + 1e-12)
        trajectories[arch_name] = traj_normed

        traj_std = traj_normed.std()
        traj_range = traj_normed.max() - traj_normed.min()
        print(f'std={traj_std:.3f}, range={traj_range:.3f}')

        if traj_std < 0.05:
            print(f'    WARNING: Very low variance, possible attractor collapse!')

    # Continue ground truth integration for N_STEPS
    print('  Ground truth (extended integration)...', end=' ')
    # Continue from the ACTUAL end state of the first trajectory
    from scipy.integrate import solve_ivp
    # The raw trajectory gives us x. Reconstruct approximate y,z from the attractor
    # For a proper comparison, integrate a fresh long trajectory
    rng_gt = np.random.default_rng(SEED + 9999)
    x0_gt = rng_gt.uniform(-15, 15)
    y0_gt = rng_gt.uniform(-15, 15)
    z0_gt = rng_gt.uniform(10, 40)
    # Integrate for seed_len + N_STEPS time points
    total_steps = SEQ_LEN + N_STEPS
    t_span_ext = (0, 25 * (total_steps / SEQ_LEN))
    t_eval_ext = np.linspace(*t_span_ext, total_steps)
    sol_ext = solve_ivp(
        _lorenz_rhs, t_span_ext, [x0_gt, y0_gt, z0_gt],
        t_eval=t_eval_ext, method='RK45', max_step=0.05
    )
    # Take the continuation portion (after the seed)
    gt_extended = sol_ext.y[0][SEQ_LEN:SEQ_LEN + N_STEPS]
    gt_trajectory_extended = (gt_extended - gmin) / (gmax - gmin + 1e-12)
    print(f'integrated ({len(gt_trajectory_extended)} steps)')

    # Use extended GT for phase portrait comparison
    gt_trajectory_for_phase = gt_trajectory_extended

    # =====================================================================
    # Save trajectory data
    # =====================================================================
    import pandas as pd

    traj_dict = {
        'time_step': np.arange(N_STEPS),
        'ground_truth': gt_trajectory_for_phase,
    }
    for _arch in ARCHITECTURES:
        if _arch in trajectories:
            traj_dict[_arch] = trajectories[_arch]
    traj_df = pd.DataFrame(traj_dict)
    traj_path = f'{RESULTS_DIR}/butterfly_trajectories_v2.csv'
    traj_df.to_csv(traj_path, index=False)
    print(f'  Saved: {traj_path}')

    # =====================================================================
    # Plotting helpers (FIX: consolidated from duplicated code)
    # =====================================================================

    TAU = 8

    def find_local_maxima(trajectory, window=5):
        '''Find indices of local maxima.'''
        maxima_indices = []
        for i in range(window, len(trajectory) - window):
            if trajectory[i] == max(trajectory[i-window:i+window+1]):
                maxima_indices.append(i)
        return np.array(maxima_indices)

    def plot_time_series(ax, traj, color, label=None, n_steps=300):
        '''Plot time series on given axes.'''
        ax.plot(traj[:n_steps], color=color, linewidth=1.5, alpha=0.8, label=label)
        ax.grid(alpha=0.2)

    def plot_phase_portrait(ax, traj, color, tau=TAU):
        '''Plot time-delay embedding phase portrait.'''
        ax.plot(traj[:-tau], traj[tau:], linewidth=0.8, alpha=0.6, color=color)
        ax.scatter(traj[0], traj[tau], c='blue', s=60, marker='o',
                   edgecolors='black', linewidth=1.2, zorder=10, label='Start')
        ax.scatter(traj[-tau-1], traj[-1], c='red', s=60, marker='s',
                   edgecolors='black', linewidth=1.2, zorder=10, label='End')
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(alpha=0.15)

    def plot_poincare(ax, traj, color):
        '''Plot Poincare return map from local maxima.'''
        maxima_idx = find_local_maxima(traj)
        if len(maxima_idx) > 1:
            max_vals = traj[maxima_idx]
            ax.scatter(max_vals[:-1], max_vals[1:], c=color, s=15, alpha=0.6,
                       edgecolors='black', linewidth=0.4)
            ax.plot(max_vals[:-1], max_vals[1:], color=color, alpha=0.25, linewidth=0.5)
        ax.grid(alpha=0.15)
        ax.set_aspect('equal', adjustable='box')

    # Prepare model data dict
    model_data = {
        'Ground Truth': {'traj': gt_trajectory_for_phase, 'color': 'black'},
    }
    for arch in ARCHITECTURES:
        if arch in trajectories:
            arch_colors = {
                'SmallBERT': '#DC2626',
                'SmallStripedHyena': '#7C3AED',
                'SmallMamba': '#16A34A',
            }
            model_data[arch] = {'traj': trajectories[arch],
                                'color': arch_colors.get(arch, '#6B7280')}

    # Save phase space and Poincare CSVs
    print('\nSaving phase space and Poincare data...')
    for model_name, data in model_data.items():
        traj = data['traj']
        safe_name = model_name.replace(' ', '_')

        phase_df = pd.DataFrame({
            'x_t': traj[:-TAU],
            f'x_t_plus_{TAU}': traj[TAU:],
        })
        phase_df.to_csv(f'{RESULTS_DIR}/butterfly_phase_{safe_name}.csv', index=False)

        maxima_idx = find_local_maxima(traj)
        if len(maxima_idx) > 1:
            max_vals = traj[maxima_idx]
            poincare_df = pd.DataFrame({
                'x_n': max_vals[:-1],
                'x_n_plus_1': max_vals[1:],
            })
            poincare_df.to_csv(f'{RESULTS_DIR}/butterfly_poincare_{safe_name}.csv', index=False)

    # =====================================================================
    # 3x3 Grid Plot (rows = test type, cols = models)
    # =====================================================================
    print('\nGenerating 3x3 grid plot...')
    n_models = len(model_data)
    fig, axes = plt.subplots(3, n_models, figsize=(6 * n_models, 16))
    model_list = list(model_data.items())
    row_labels = ['Time Series', 'Phase Portrait', 'Poincare Map']

    for col_idx, (model_name, data) in enumerate(model_list):
        traj = data['traj']
        color = data['color']

        # Row 0: Time Series
        ax = axes[0, col_idx]
        plot_time_series(ax, traj, color)
        ax.set_title(model_name, fontweight='bold', fontsize=11)
        if col_idx == 0:
            ax.set_ylabel('Time Series\nValue', fontsize=10, fontweight='bold')

        # Row 1: Phase Portrait
        ax = axes[1, col_idx]
        plot_phase_portrait(ax, traj, color)
        ax.set_ylabel(f'x(t+{TAU})', fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(f'Phase Portrait\nx(t+{TAU})', fontsize=10, fontweight='bold')

        # Row 2: Poincare Map
        ax = axes[2, col_idx]
        plot_poincare(ax, traj, color)
        ax.set_xlabel('x_n (peak n)', fontsize=10)
        ax.set_ylabel('x_{n+1}', fontsize=10)
        if col_idx == 0:
            ax.set_ylabel('Poincare Map\nx_{n+1}', fontsize=10, fontweight='bold')

    fig.suptitle('Phase Portrait Reconstruction: Phase Portrait Reconstruction
                 'Autoregressive generation on Lorenz attractor',
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0.02, 0, 1, 0.97])

    fig_path = f'{RESULTS_DIR}/butterfly_test_3x3_grid_v2.png'
    plt.savefig(fig_path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {fig_path}')
    plt.show()

    # =====================================================================
    # Verdict
    # =====================================================================
    print('\n' + '=' * 70)
    print('PHASE PORTRAIT VERDICT')
    print('=' * 70)

    gt_phase_var = np.var(gt_trajectory_for_phase[:-TAU]) + np.var(gt_trajectory_for_phase[TAU:])

    for arch_name in ARCHITECTURES:
        if arch_name not in trajectories:
            continue
        traj = trajectories[arch_name]
        phase_var = np.var(traj[:-TAU]) + np.var(traj[TAU:])
        ratio = phase_var / (gt_phase_var + 1e-8)

        print(f'\n{arch_name}:')
        print(f'  Phase space variance: {phase_var:.4f}')
        print(f'  GT variance ratio:    {ratio:.4f}')

        if ratio < 0.3:
            print(f'  COLLAPSED: Model attractor collapsed (variance < 30% of GT)')
        elif ratio < 0.6:
            print(f'  DAMPED: Overly damped (30-60% of GT)')
        elif ratio > 1.5:
            print(f'  DIVERGENT: Unstable (> 150% of GT)')
        else:
            print(f'  PRESERVED: Maintains attractor structure (60-150% of GT)')

    # =====================================================================
    # LLE + Seed Ablation
    # =====================================================================

    def estimate_largest_lyapunov(trajectory, tau=1, m=3, max_iter=None):
        '''Estimate the Largest Lyapunov Exponent using Rosenstein method.'''
        N = len(trajectory)
        if N < (m - 1) * tau + 50:
            return float('nan')
        n_points = N - (m - 1) * tau
        if max_iter is None:
            max_iter = n_points // 4
        X = np.array([trajectory[i:i + m * tau:tau] for i in range(n_points)])
        min_gap = max(2 * tau, 10)
        nn_indices = np.zeros(n_points, dtype=int)
        for i in range(n_points):
            dists = np.linalg.norm(X - X[i], axis=1)
            dists[max(0, i - min_gap):min(n_points, i + min_gap + 1)] = np.inf
            nn_indices[i] = np.argmin(dists)
        divergences = []
        for k in range(1, max_iter):
            log_divs = []
            for i in range(n_points - k):
                j = nn_indices[i]
                if j + k < n_points:
                    d = np.linalg.norm(X[i + k] - X[j + k])
                    if d > 0:
                        log_divs.append(np.log(d))
            if log_divs:
                divergences.append(np.mean(log_divs))
        if len(divergences) < 10:
            return float('nan')
        divergences = np.array(divergences)
        t = np.arange(1, len(divergences) + 1)
        n_fit = max(5, len(t) // 4)
        slope, _ = np.polyfit(t[:n_fit], divergences[:n_fit], 1)
        return slope

    print('\n' + '=' * 70)
    print('DYNAMICAL SYSTEMS DIAGNOSTICS')
    print('=' * 70)

    print('\nLargest Lyapunov Exponent (LLE) Estimates:')
    print('  (Positive = chaos, Zero/Negative = periodic/convergent)')

    lle_gt = estimate_largest_lyapunov(gt_trajectory_for_phase, tau=TAU, m=3)
    print(f'\n  Ground Truth (Lorenz): LLE = {lle_gt:.4f}', end='')
    print('  --> CHAOTIC' if lle_gt > 0.001 else '  --> non-chaotic')

    for arch_name in ARCHITECTURES:
        if arch_name not in trajectories:
            continue
        traj = trajectories[arch_name]
        lle = estimate_largest_lyapunov(traj, tau=TAU, m=3)
        print(f'  {arch_name:20s}: LLE = {lle:.4f}', end='')
        if lle > 0.001:
            print('  --> CHAOTIC (preserves sensitivity)')
        elif abs(lle) < 0.001:
            print('  --> PERIODIC (limit cycle collapse)')
        else:
            print('  --> CONVERGENT (point attractor)')

    # =====================================================================
    # Seed Ablation
    # =====================================================================
    print('\n' + '=' * 70)
    print('SEED ABLATION: Lorenz Phase Portrait')
    print(f'Testing {len(SEEDS)} seeds to verify collapse is architectural')
    print('=' * 70)

    seed_results = {arch: [] for arch in ARCHITECTURES}

    for ablation_seed in SEEDS:
        for arch_name in ARCHITECTURES:
            ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_seed{ablation_seed}_best.pt'
            if not os.path.exists(ckpt_path):
                continue

            # Detect checkpoint vocab size to handle legacy (257) vs current (258)
            ckpt_state = torch.load(ckpt_path, map_location=DEVICE)
            ckpt_vocab = ckpt_state['tok_emb.weight'].shape[0]
            abl_model = MODEL_CLASSES[arch_name](vocab_size=ckpt_vocab).to(DEVICE)
            abl_model.load_state_dict(ckpt_state)

            torch.manual_seed(GENERATION_SEED)
            abl_traj = generate_autoregressive(abl_model, seed_seq, steps=N_STEPS, temperature=TEMP)
            abl_continuous = undiscretize(abl_traj, global_range=gt_range)
            abl_normed = (abl_continuous - gmin) / (gmax - gmin + 1e-12)

            phase_var = np.var(abl_normed[:-TAU]) + np.var(abl_normed[TAU:])
            ratio = phase_var / (gt_phase_var + 1e-8)
            abl_lle = estimate_largest_lyapunov(abl_normed, tau=TAU, m=3)

            seed_results[arch_name].append({
                'seed': ablation_seed,
                'variance': phase_var,
                'gt_ratio': ratio,
                'std': np.std(abl_normed),
                'lle': abl_lle,
            })

            del abl_model
            cleanup_gpu()

    # Print seed ablation summary
    print('\n' + '-' * 60)
    seed_rows = []
    for arch_name in ARCHITECTURES:
        results = seed_results[arch_name]
        if not results:
            print(f'\n  {arch_name}: No seed ablation data')
            continue

        ratios = [r['gt_ratio'] for r in results]
        stds = [r['std'] for r in results]
        lles = [r['lle'] for r in results if not np.isnan(r.get('lle', float('nan')))]

        print(f'\n  {arch_name} ({len(results)} seeds):')
        print(f'    GT variance ratio: {np.mean(ratios):.4f} +/- {np.std(ratios):.4f}')
        print(f'    Trajectory std:    {np.mean(stds):.4f} +/- {np.std(stds):.4f}')
        if lles:
            print(f'    Mean LLE:          {np.mean(lles):.4f} +/- {np.std(lles):.4f}')

        collapsed = sum(1 for r in ratios if r < 0.3)
        preserved = sum(1 for r in ratios if 0.6 <= r <= 1.5)
        print(f'    Collapsed: {collapsed}/{len(results)}  Preserved: {preserved}/{len(results)}')

        for r in results:
            seed_rows.append({
                'architecture': arch_name,
                'seed': r['seed'],
                'phase_variance': r['variance'],
                'gt_ratio': r['gt_ratio'],
                'trajectory_std': r['std'],
                'lle': r.get('lle', float('nan')),
            })

    if seed_rows:
        df_seeds = pd.DataFrame(seed_rows)
        df_seeds.to_csv(f'{RESULTS_DIR}/butterfly_seed_ablation_v2.csv', index=False)
        print(f'\n  Saved seed ablation CSV')

    # Verdict
    bert_ratios = [r['gt_ratio'] for r in seed_results.get('SmallBERT', [])]
    mamba_ratios = [r['gt_ratio'] for r in seed_results.get('SmallMamba', [])]

    if bert_ratios and mamba_ratios:
        bert_collapse_rate = sum(1 for r in bert_ratios if r < 0.3) / len(bert_ratios)
        mamba_preserve_rate = sum(1 for r in mamba_ratios if r >= 0.6) / len(mamba_ratios)

        print('\n' + '=' * 60)
        print('SEED ABLATION VERDICT:')
        if bert_collapse_rate > 0.7 and mamba_preserve_rate > 0.5:
            print(f'  Transformer collapses in {bert_collapse_rate:.0%} of seeds')
            print(f'  SSM preserves in {mamba_preserve_rate:.0%} of seeds')
            print(f'  --> Limit-cycle collapse is ARCHITECTURAL, not a random seed artifact')
        elif bert_collapse_rate > 0.3:
            print(f'  Transformer collapses in {bert_collapse_rate:.0%} of seeds')
        else:
            print(f'  Transformer only collapses in {bert_collapse_rate:.0%} of seeds')
        print('=' * 60)

else:
    print('Skipping phase portrait generation (missing checkpoints).')
